# Part 2: Bivariate & Multivariate Analysis — 15 Business Questions (Notebook 2)

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_dark"

In [2]:
# Load the cleaned dataset saved in Part 1
df_sales = pd.read_csv('online_retail_cleaned_2.csv')
df_sales['InvoiceDate'] = pd.to_datetime(df_sales['InvoiceDate'])

### Q1: Which top 10 countries generate the highest total sales revenue?

In [3]:
q1_data = df_sales.groupby('Country')['TotalAmount'].sum().sort_values(ascending=False).head(10).reset_index()
fig1 = px.bar(
    q1_data, x='TotalAmount', y='Country', orientation='h',
    title='Q1: Top 10 Countries by Total Sales Revenue',
    labels={'TotalAmount': 'Total Revenue ($)', 'Country': 'Country'},
    color='TotalAmount', color_continuous_scale='Viridis'
)
fig1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig1.show()

### Q2: What is the monthly revenue trend over the dataset's time frame?

In [4]:
q2_data = df_sales.groupby('YearMonth')['TotalAmount'].sum().reset_index()
fig2 = px.line(
    q2_data, x='YearMonth', y='TotalAmount', markers=True,
    title='Q2: Monthly Sales Revenue Trend',
    labels={'YearMonth': 'Year-Month', 'TotalAmount': 'Revenue ($)'},
    color_discrete_sequence=['#00E5FF']
)
fig2.show()

### Q3: Which days of the week generate the highest total sales revenue?

In [5]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Sunday']
q3_data = df_sales.groupby('DayOfWeek')['TotalAmount'].sum().reindex(day_order).reset_index()
fig3 = px.bar(
    q3_data, x='DayOfWeek', y='TotalAmount',
    title='Q3: Revenue Distribution by Day of Week',
    labels={'DayOfWeek': 'Day of Week', 'TotalAmount': 'Revenue ($)'},
    color='TotalAmount', color_continuous_scale='Magma'
)
fig3.show()

### Q4: What peak hours of the day receive the highest number of unique Invoices?

In [6]:
q4_data = df_sales.groupby('Hour')['InvoiceNo'].nunique().reset_index()
fig4 = px.area(
    q4_data, x='Hour', y='InvoiceNo',
    title='Q4: Peak Hourly Volume of Unique Invoices',
    labels={'Hour': 'Hour of Day (24h)', 'InvoiceNo': 'Unique Invoices'},
    color_discrete_sequence=['#FF6600']
)
fig4.show()

### Q5: What are the top 10 best-selling products by overall quantity sold?

In [7]:
q5_data = df_sales.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10).reset_index()
fig5 = px.bar(
    q5_data, x='Quantity', y='Description', orientation='h',
    title='Q5: Top 10 Best-Selling Products by Quantity',
    labels={'Quantity': 'Total Quantity Sold', 'Description': 'Product'},
    color='Quantity', color_continuous_scale='Purples'
)
fig5.update_layout(yaxis={'categoryorder': 'total ascending'})
fig5.show()

### Q6: What are the top 10 revenue-generating products?

In [8]:
q6_data = df_sales.groupby('Description')['TotalAmount'].sum().sort_values(ascending=False).head(10).reset_index()
fig6 = px.bar(
    q6_data, x='TotalAmount', y='Description', orientation='h',
    title='Q6: Top 10 Revenue-Generating Products',
    labels={'TotalAmount': 'Total Revenue ($)', 'Description': 'Product'},
    color='TotalAmount', color_continuous_scale='Magma'
)
fig6.update_layout(yaxis={'categoryorder': 'total ascending'})
fig6.show()

### Q7: What is the relationship between Unit Price and Quantity sold per transaction?

In [9]:
sample_df = df_sales[(df_sales['Quantity'] < 500) & (df_sales['UnitPrice'] < 50)].sample(2000, random_state=42)
fig7 = px.scatter(
    sample_df, x='UnitPrice', y='Quantity', color='TotalAmount',
    title='Q7: Bivariate Scatter Plot: Unit Price vs. Quantity',
    labels={'UnitPrice': 'Unit Price ($)', 'Quantity': 'Quantity Sold'},
    color_continuous_scale='Rainbow', opacity=0.7
)
fig7.show()

### Q8: What is the average spend per individual invoice across the top 10 countries?

In [10]:
top10_countries = q1_data['Country'].tolist()
q8_df = df_sales[df_sales['Country'].isin(top10_countries)]
invoice_spend = q8_df.groupby(['Country', 'InvoiceNo'])['TotalAmount'].sum().reset_index()
q8_data = invoice_spend.groupby('Country')['TotalAmount'].mean().reset_index()

fig8 = px.bar(
    q8_data, x='Country', y='TotalAmount',
    title='Q8: Average Spend per Invoice Across Top 10 Countries',
    labels={'TotalAmount': 'Average Invoice Value ($)', 'Country': 'Country'},
    color='TotalAmount', color_continuous_scale='Purples'
)
fig8.show()

### Q9: Who are the top 10 highest-spending customers?

In [11]:
q9_data = df_sales.groupby('CustomerID')['TotalAmount'].sum().sort_values(ascending=False).head(10).reset_index()
fig9 = px.bar(
    q9_data, x='CustomerID', y='TotalAmount',
    title='Q9: Top 10 Customers by Total Spend',
    labels={'CustomerID': 'Customer ID', 'TotalAmount': 'Total Spend ($)'},
    color='TotalAmount', color_continuous_scale='Electric'
)
fig9.show()

### Q10: How are total invoice values distributed across transactions?

In [12]:
inv_totals = df_sales.groupby('InvoiceNo')['TotalAmount'].sum().reset_index()
inv_filtered = inv_totals[inv_totals['TotalAmount'] < 1000]

fig10 = px.histogram(
    inv_filtered, x='TotalAmount', nbins=50,
    title='Q10: Distribution of Total Spend per Invoice (< $1,000)',
    labels={'TotalAmount': 'Invoice Value ($)'},
    color_discrete_sequence=['purple']
)
fig10.show()

### Q11: How many unique invoices are generated by top repeat customers?

In [13]:
q11_data = df_sales.groupby('CustomerID')['InvoiceNo'].nunique().sort_values(ascending=False).head(10).reset_index()
fig11 = px.bar(
    q11_data, x='CustomerID', y='InvoiceNo',
    title='Q11: Top 10 Customers by Total Order Count (Unique Invoices)',
    labels={'CustomerID': 'Customer ID', 'InvoiceNo': 'Unique Order Count'},
    color='InvoiceNo', color_continuous_scale='Electric'
)
fig11.show()

### Q12: How does order volume heat up across days of the week and hours of the day?

In [14]:
heatmap_data = df_sales.groupby(['DayOfWeek', 'Hour'])['InvoiceNo'].nunique().unstack().reindex(day_order)
fig12 = px.imshow(
    heatmap_data,
    labels=dict(x="Hour of Day", y="Day of Week", color="Invoice Count"),
    title="Q12: Heatmap of Order Volume (Day of Week vs Hour of Day)",
    color_continuous_scale="Hot"
)
fig12.show()

### Q13: What is the monthly average order value (AOV) trend over time?

In [15]:
monthly_inv = df_sales.groupby('YearMonth').agg(
    TotalRev=('TotalAmount', 'sum'),
    UniqueInvoices=('InvoiceNo', 'nunique')
).reset_index()
monthly_inv['AOV'] = monthly_inv['TotalRev'] / monthly_inv['UniqueInvoices']

fig13 = px.line(
    monthly_inv, x='YearMonth', y='AOV', markers=True,
    title='Q13: Monthly Average Order Value (AOV) Trend',
    labels={'YearMonth': 'Year-Month', 'AOV': 'Average Order Value ($)'},
    color_discrete_sequence=['#39FF14']
)
fig13.show()

### Q14: What proportion of total revenue comes from the top 10% of customers? (Pareto Analysis)

In [16]:
cust_rev = df_sales.groupby('CustomerID')['TotalAmount'].sum().sort_values(ascending=False).reset_index()
cust_rev['CumulativeRevenue'] = cust_rev['TotalAmount'].cumsum()
cust_rev['CumulativePercent'] = (cust_rev['CumulativeRevenue'] / cust_rev['TotalAmount'].sum()) * 100
cust_rev['CustomerRankPercent'] = (np.arange(len(cust_rev)) + 1) / len(cust_rev) * 100

fig14 = px.line(
    cust_rev, x='CustomerRankPercent', y='CumulativePercent',
    title='Q14: Cumulative Revenue Contribution by Customer Percentage (Pareto Principle)',
    labels={'CustomerRankPercent': '% of Total Customers', 'CumulativePercent': '% of Total Revenue'},
    color_discrete_sequence=['#FFD700']
)
fig14.add_hline(y=80, line_dash="dash", line_color="red", annotation_text="80% Revenue Benchmark")
fig14.show()

### Q15: How does overall transaction volume compare between the UK and international markets?

In [17]:
df_sales['Market'] = np.where(df_sales['Country'] =='United Kingdom' , 'United Kingdom' , 'International')
q15_data = df_sales.groupby('Market')['TotalAmount'].sum().reset_index()

fig15 = px.pie(
    q15_data , values='TotalAmount' , names='Market', hole=0.4,
    title = 'Q15: Revenue Share: United Kingdom Vs International Markets',
    color_discrete_sequence=['#007ACC','#FF4136']
)
fig15.show()

# The END